# Drivers, passengers and mutational meltdown

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sisyga/biolgca/blob/aidevelop/docs/source/zoo/mutational_meltdown.ipynb)

*Syga S (2023). Evolutionary cellular automata for tumor growth and
invasion. PhD thesis, TU Dresden, chapter "The interplay of invasion and
mutational meltdown". Code:
[doi:10.5281/zenodo.10014813](https://doi.org/10.5281/zenodo.10014813).
Parameters after McFarland CD, Mirny LA, Korolev KS (2014), PNAS 111:
15138–15143.*

**Question.** Every division can bring a mutation. Rarely it is a *driver*
that makes the cell divide much faster; far more often a *passenger* that
slows it down a little. Passengers are too weak for selection to remove
them, so they accumulate, and a small tumour can decline and die out from
its own mutational load: a *mutational meltdown*, driven by Muller's
ratchet. When does the tumour win this tug-of-war, and can it escape the
ratchet by invading the surrounding tissue?

In [ ]:
# In Google Colab this cell installs BioLGCA (about a minute); elsewhere it does nothing.
import importlib.util
import subprocess
import sys

if "google.colab" in sys.modules and importlib.util.find_spec("lgca") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "biolgca @ git+https://github.com/sisyga/biolgca@aidevelop"], check=True)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import lgca
from lgca import zoo
from lgca.zoo import mutational_meltdown as meltdown

zoo.parameter_table("mutational_meltdown")

## The model

Identity-based cells without volume exclusion on a 1D lattice; every cell
has its own proliferation rate $\alpha$. In every time step:

1. **Birth and death:** every cell dies with probability $\delta$ and
   divides with probability $\alpha(1 - n/K)$.
2. **Mutations:** a daughter acquires a passenger with probability
   $p_p = 0.05$, which lowers its $\alpha$ by an exponentially distributed
   amount of mean $\alpha_0 s_p = 5 \cdot 10^{-4}$, and a driver with
   probability $p_d = 7 \cdot 10^{-6}$, which raises it by an exponentially
   distributed amount of mean $\alpha_0 s_d = 0.1$ ($\alpha$ stays within
   $[0, 1]$).
3. **Migration:** every cell takes a channel at random, a velocity channel
   with weight 1 or the rest channel with weight $e^\gamma$, so it moves
   with probability $2/(2 + e^\gamma)$.

The probabilities come from McFarland et al.: a mutation probability of
$10^{-8}$ per locus and division, $T_d = 700$ driver loci and
$T_p = 5 \cdot 10^6$ passenger loci.

**The critical size.** In a population of $N$ cells at equilibrium, a
passenger is nearly neutral and fixes with probability $1/N$, a driver with
probability about $s_d$. Weighing how often each arises, fixes and changes
the population gives a mean change of the population size
$\propto (N/N^* - 1)$ with

$$N^* = \frac{T_p\, s_p}{T_d\, s_d^2},$$

which is about 179 cells here. Below $N^*$ passengers win and the
population shrinks, which makes passengers win even more; above it drivers
win.

In [ ]:
print(f"N* = {meltdown.critical_size():.1f} cells")

## The tug-of-war in isolated populations

As in the thesis (and McFarland et al.): populations of $N_0 = N^*/2$,
$N^*$ and $2N^*$ cells at the equilibrium of a node with capacity
$K = 2N_0$ ($\alpha_0 = 0.5$, $\delta = 0.25$), each in a node of its own that
exchanges no cells with the others (`meltdown.isolated_nodes`). Four
populations of each size, as in the thesis figure, for 40 000 steps, i.e. $10^4$ generations of
$1/\delta = 4$ steps (about 20 s):

In [ ]:
histories = {n0: meltdown.node_histories(meltdown.isolated_nodes(n0, nodes=4), every=200) for n0 in (0.5, 1.0, 2.0)}

fig, axes = plt.subplots(2, 1, figsize=(7, 7), sharex=True, constrained_layout=True)
for colour, (n0, history) in zip(("tab:blue", "tab:orange", "tab:green"), histories.items()):
    generations = history["steps"] * 0.25
    axes[0].plot(generations, history["cells"], color=colour, alpha=0.7)
    axes[1].plot(generations, history["alpha"], color=colour, alpha=0.7)
    axes[1].plot([], [], color=colour, label=f"N₀ = {n0:g} N*")
axes[0].axhline(meltdown.critical_size(), color="k", ls="--", lw=1, label="N*")
axes[0].set(ylabel="population size $N$", yscale="log", ylim=(10, 700))
axes[0].legend()
axes[1].set(xlabel="time (generations)", ylabel="mean proliferation rate α", ylim=(0.2, 1))
axes[1].legend()
plt.show()
plt.close(fig)

for n0, history in histories.items():
    print(f"N₀ = {n0:g} N*: {np.count_nonzero(history['cells'][-1] == 0)} of {history['cells'].shape[1]} populations extinct, "
          f"mean α of the others {np.nanmean(history['alpha'][-1]):.2f}")

As in the thesis:

- **Below $N^*$** (blue) the mean proliferation rate falls steadily as
  passengers accumulate, the population shrinks with it, and most
  populations die out within $10^4$ generations. A driver that fixes (a
  jump in $\alpha$) can rescue a population, but it is rare in a small one.
- **At $N^*$** (orange) drivers and passengers balance on average: the
  populations wander, and their fates differ from run to run.
- **Above $N^*$** (green) drivers fix more often, and the populations grow
  towards the capacity allowed by $\alpha \to 1$, $K(1 - \delta)$. Between
  drivers, passengers pull $\alpha$ down in every population.

## Invasion as an escape

Now the tumour is not confined: it starts at the left end of a 1D tissue
and can spread into it, one node of capacity $K$ after the other. As in the
thesis code of this part, $\delta = 0.375$ and $K = N_0/(1 - \delta/\alpha_0)$,
so that $N_0$ is the equilibrium of one node. A run ends when the tumour
dies out (*extinct*) or reaches $\max(2N_0, N^*)$ cells (*cancer*). Four
runs each for three rest-channel weights $\gamma$, i.e. fast, slow and
almost no movement, with $N_0 = N^*/4$:

In [ ]:
colours = {0.0: "tab:green", 8.0: "tab:orange", 16.0: "tab:blue"}
fig, axes = plt.subplots(3, 1, figsize=(7, 8), sharex=True, constrained_layout=True)
for gamma, colour in colours.items():
    for seed in range(4):
        spec = meltdown.build_spec(n0=0.25, gamma=gamma, seed=seed)
        result, steps, history = meltdown.outcome(spec, every=10, history=True)
        generations = np.maximum(history["steps"], 1) * 0.375
        axes[0].plot(generations, history["cells"], color=colour, alpha=0.7)
        axes[1].plot(generations, history["extent"], color=colour, alpha=0.7)
        axes[2].plot(generations, history["alpha"], color=colour, alpha=0.7)
    axes[2].plot([], [], color=colour, label=f"γ = {gamma:g}, moving {2 / (2 + np.exp(gamma)):.0e}")
axes[0].axhline(meltdown.critical_size(), color="k", ls="--", lw=1)
axes[0].set(ylabel="population size $N$", yscale="log")
axes[1].set(ylabel="occupied nodes")
axes[2].set(xlabel="time (generations)", ylabel="mean α", xscale="log")
axes[2].legend(fontsize="small")
plt.show()
plt.close(fig)

With fast movement (green) the tumour spreads into free nodes, where cells
divide freely, and passes $N^*$ in a few dozen generations. With slow
movement (orange) it waits at its node until a cell founds a new one and
then grows. Cells that barely move (blue) stay in their node, whose
population is far below $N^*$, and die out.

## When does a tumour escape?

The fraction of runs that end as cancer for initial sizes from $N^*/20$ to
$N^*$ and rest-channel weights from 0 to 16 (the thesis: a 21 × 21 grid of
100 runs each; here 5 × 5 × 6 runs on four processes, runs limited to
20 000 steps; about 30 s). Runs that neither die out nor reach the target
within the limit count as not cancer:

In [ ]:
n0s, gammas = [0.05, 0.1, 0.25, 0.5, 1.0], [0, 4, 8, 12, 16]
table = meltdown.scan(n0s, gammas, seeds=range(6), n_jobs=4)
cancer = (table.assign(cancer=table["outcome"] == "cancer")
          .pivot_table(index="n0", columns="gamma", values="cancer", aggfunc="mean"))

fig, axis = plt.subplots(figsize=(6, 4))
image = axis.imshow(cancer.values, origin="lower", cmap="coolwarm", vmin=0, vmax=1, aspect="auto")
axis.set(xticks=range(len(gammas)), xticklabels=gammas, yticks=range(len(n0s)), yticklabels=n0s,
         xlabel="rest-channel weight γ", ylabel="initial size N₀ / N*")
fig.colorbar(image, label="probability of cancer")
plt.show()
plt.close(fig)

As in the thesis (its Fig 5.3): a tumour that is large enough, or that
moves fast enough, grows; a small tumour whose cells barely move dies out,
and the larger the tumour, the less motility it needs to escape.

But what kills the small, immobile tumours? The same scan for the columns
of slow movement, with four runs each, once more without any mutations
($p_p = p_d = 0$):

In [ ]:
subset = {"n0s": [0.1, 0.5], "gammas": [12, 16], "seeds": range(4), "n_jobs": 4}
control = meltdown.scan(**subset, p_p=0.0, p_d=0.0)
with_mutations = table[table["n0"].isin(subset["n0s"]) & table["gamma"].isin(subset["gammas"])
                       & table["seed"].isin(subset["seeds"])]


def summary(outcomes):
    return " ".join(f"{name} {np.count_nonzero(outcomes == name)}" for name in ("cancer", "extinct", "undecided"))


for n0 in subset["n0s"]:
    for gamma in subset["gammas"]:
        pick = lambda frame: frame[(frame["n0"] == n0) & (frame["gamma"] == gamma)]["outcome"].to_numpy()
        print(f"N₀ = {n0:g} N*, γ = {gamma:2d}:  with mutations {summary(pick(with_mutations)):31s}"
              f" without: {summary(pick(control))}")

Two different deaths:

- **Small tumours die out without any mutation.** At $N_0 = N^*/10$ a node
  holds 18 cells, and with $\delta = 0.375$ against $\alpha_0 = 0.5$ such a
  small population dies out by chance within a few thousand steps, long
  before passengers matter. Invasion rescues it only if cells leave and
  found new nodes before that happens; a founder cell starts a new node
  with probability $1 - \delta/\alpha_0 = 1/4$.
- **Tumours near $N^*$ die of their mutations.** At $N_0 = N^*/2$ a tumour
  without mutations persists or grows (undecided: still alive after 20 000
  steps), while with passengers it melts down. This is the ratchet, and
  here invasion is an escape from it.

So the lower left of the diagram is demographic extinction, and only its
upper right is mutational meltdown. The thesis interprets the whole
diagram as meltdown.

## Differences from the thesis

- **Death rate.** The isolated populations use $\delta = 0.25$ as the
  thesis text says; the invasion runs use $\delta = 0.375$, the value in the
  thesis code that made the probability-of-cancer diagram. With
  $\delta = 0.25$ founders survive more often, and small tumours escape
  at larger $\gamma$ than in the thesis figure.
- **The scan** is coarse (5 × 5 × 6 runs instead of 21 × 21 × 100) and
  limits runs to 20 000 steps (`build_spec(full=True)`: 100 000).

## Explore it

A tumour of $N^*/2$ cells with slow movement, with the density per node
over time. Move $\gamma$ and the passenger probability $p_p$ while it runs:

In [ ]:
explorer = lgca.explore(
    meltdown.build_spec(n0=0.5, gamma=10.0),
    {"dynamics.operators[1].terms[resting_bias].beta": (0.0, 16.0),  # γ
     "dynamics.operators[birth_death].mutation[0].probability": (0.0, 0.2)},  # p_p
    measure=["population", "r_b"],
    steps_per_frame=20,
)
explorer

## Things to try

1. **The mutation rate.** McFarland et al. find a critical mutation rate
   above which tumours melt down at any size. Raise $p_l$ (both $p_p$ and
   $p_d$ scale with it) in isolated populations of $2N^*$ cells.
2. **Stronger passengers.** $N^*$ grows with $s_p$: double it and repeat
   the tug-of-war.
3. **Two dimensions.** Invasion into a plane gives a tumour more new
   nodes at its edge than a line: run the model on a square lattice
   (`build_spec` is a `ModelSpec`; change `space` and the initial nodes).
4. **The founder effect.** Measure how often a single cell placed in an
   empty node founds a population that reaches $K(1 - \delta/\alpha_0)$
   cells, with and without mutations, and compare with
   $1 - \delta/\alpha_0$.